In [ ]:
#@title Settings

branch = 'main' #@param {type:"string"}
prompt = 'A serene mountain lake at sunrise with mist rising over calm water, cinematic lighting' #@param {type:"string"}
width = 448 #@param {type:"integer"}
height = 256 #@param {type:"integer"}
frames = 17 #@param {type:"integer"}
steps = 15 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
train_lora = False #@param {type:"boolean"}
train_steps = 3 #@param {type:"integer"}

In [ ]:
#@title GPU

!nvidia-smi
import torch
cc = torch.cuda.get_device_capability(0)
print(torch.__version__, torch.cuda.get_device_name(0), cc)
assert cc >= (8, 0), f'SM80+ GPU required (A100/L4), got SM{cc[0]}{cc[1]}'

In [ ]:
#@title Setup

%cd /content
!git clone --depth 1 -b {branch} https://github.com/seruva19/mirai.git
%cd /content/mirai

import tomllib
deps = tomllib.load(open('pyproject.toml', 'rb'))['project']['dependencies']
open('/tmp/mirai_deps.txt', 'w').write('\n'.join(d for d in deps if not d.startswith(('torch==', 'torchvision=='))))
!pip install -q -r /tmp/mirai_deps.txt
!pip install -q -e . --no-deps

In [ ]:
#@title Download model

!python -u scripts/download.py --variant lingbot-video-moe-30b-a3b --output-dir models/lingbot-video-moe-30b-a3b

In [ ]:
#@title Generate

!python -u scripts/infer.py \
  --config configs/lingbot_video/inference_nf4_32gb.toml \
  --prompt "{prompt}" \
  --width {width} \
  --height {height} \
  --frames {frames} \
  --steps {steps} \
  --seed {seed} \
  --out outputs/demo_clip.mp4

from IPython.display import Video, display
display(Video('outputs/demo_clip.mp4', embed=True))

In [ ]:
#@title Train LoRA (optional)

if train_lora:
  import os
  import shutil
  os.makedirs('data/demo_videos', exist_ok=True)
  shutil.copy2('outputs/demo_clip.mp4', 'data/demo_videos/sample.mp4')
  open('data/demo_videos/sample.txt', 'w').write(prompt + '\n')

  open('configs/colab_demo_train.toml', 'w').write(f'''preset = "lingbot_video"
[model]
path = "./models/lingbot-video-moe-30b-a3b"
hash_snapshot_contents = false
[training]
batch_size = 1
gradient_accumulation = 1
max_steps = {train_steps}
gradient_checkpointing = "aggressive"
block_swap_mode = "async"
blocks_to_swap = 20
[optimizer]
type = "paged_adamw_8bit"
allow_fallback = false
[adapter]
type = "lora"
target_preset = "attn_router_routed_experts"
rank = 16
alpha = 16.0
[dataset]
path = "./data/demo_videos"
auto_preprocess_cache = true
preprocess_raw_media_to_pt = true
enable_bucketing = true
frame_buckets = [{frames}]
bucket_resolutions = ["{height}x{width}"]
[memory]
frozen_weight_quantization = "nf4"
frozen_weight_quantization_strategy = "compressed_weights"
quantize_experts_on_load = true
expert_weight_access = "chunked_dequant"
expert_dequant_chunk_size = 32
weight_residency_strategy = "block_swap"
block_swap_prefetch_depth = 1
block_residency_planner = "phase_aware"
block_swap_transfer_strategy = "flat_ring"
max_pinned_host_gib = 16.0
cuda_memory_fraction = 0.95
minimum_system_memory_gib = 8.0
[logging]
output_dir = "./outputs/colab_demo_lora"
save_every_n_steps = {train_steps}
sample_every_n_steps = 0
''')
  !python -u scripts/train.py --config configs/colab_demo_train.toml

  infer_cfg = open('configs/lingbot_video/inference_nf4_32gb.toml').read()
  infer_cfg += '\n[adapter]\ntype = "lora"\ntarget_preset = "attn_router_routed_experts"\nrank = 16\nalpha = 16.0\n'
  open('configs/colab_demo_infer_lora.toml', 'w').write(infer_cfg)
  !python -u scripts/infer.py --config configs/colab_demo_infer_lora.toml --adapter outputs/colab_demo_lora/adapter.safetensors --prompt "{prompt}" --width {width} --height {height} --frames {frames} --steps {steps} --seed {seed} --out outputs/demo_clip_lora.mp4
  from IPython.display import Video, display
  display(Video('outputs/demo_clip_lora.mp4', embed=True))